# 🚀 Fase 3: Modelado de Machine Learning (XGBoost & Detección de Fraude)

En este segundo cuaderno conectaremos la arquitectura modular desarrollada en la carpeta `src/`. Vamos a:
1. Cargar los datos crudos usando `data_loader.py`.
2. Aplicar el filtrado de canales de riesgo e ingeniería de características usando `preprocessing.py`.
3. Preparar la matriz de entrenamiento y entrenar un modelo **XGBoost** optimizado para manejar desbalanceo de clases.
4. Evaluar el rendimiento utilizando métricas orientadas a negocio (**PR-AUC** y **Recall**).

In [1]:
# Importar librerías de manipulación y modelado
import pandas as pd
import numpy as np
import sys
import os

# Asegurar que Python reconozca la carpeta src/ en el path
sys.path.append(os.path.abspath("../src"))

from data_loader import load_raw_data
from preprocessing import filter_relevant_transactions, engineer_features
from train import prepare_modeling_data

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, precision_recall_curve, auc, confusion_matrix

print("Librerías y módulos importados correctamente.")

Librerías y módulos importados correctamente.


### 1. Ejecución del Pipeline Modular de Preprocesamiento
Invocamos nuestras funciones personalizadas para cargar el dataset, eliminar los canales con 0% de fraude (`PAYMENT`, `CASH_IN`, `DEBIT`) y calcular los errores de discrepancia en los saldos.

In [2]:
# Cargar datos crudos
PATH = "../data/raw/PS_20174392719_1491204439457_log.csv"
df_raw = load_raw_data(PATH)

# Aplicar filtrado y características
df_filtered = filter_relevant_transactions(df_raw)
df_processed = engineer_features(df_filtered)

# Preparar X e y para Machine Learning
X, y = prepare_modeling_data(df_processed)

print(f"\nDimensiones finales de la matriz de características (X): {X.shape}")
print(f"Distribución de la variable objetivo (y):")
print(y.value_counts(normalize=True) * 100)

Dataset cargado con éxito. Dimensiones: 6,362,620 filas y 11 columnas.
Dimensiones iniciales del dataset: 6,362,620 filas.
Dimensiones tras filtrar canales de riesgo: 2,770,409 filas.
Iniciando ingeniería de características (feature engineering)...

Dimensiones finales de la matriz de características (X): (2770409, 7)
Distribución de la variable objetivo (y):
isFraud
0    99.703546
1     0.296454
Name: proportion, dtype: float64


> **💡 Criterio de Partición:**
> Dado que estamos tratando con datos financieros secuenciales, dividiremos el dataset asegurando una estratificación adecuada para conservar la proporción ultra minoritaria de los fraudes tanto en el conjunto de entrenamiento como en el de prueba.

In [3]:
# División Train-Test Estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]:,} registros")
print(f"Conjunto de prueba: {X_test.shape[0]:,} registros")

Conjunto de entrenamiento: 2,216,327 registros
Conjunto de prueba: 554,082 registros


### 2. Entrenamiento del Modelo XGBoost con Manejo de Desbalanceo
Para enfrentar la asimetría severa de las clases (menos del 0.2% de fraudes en los canales filtrados), calcularemos el parámetro `scale_pos_weight` para penalizar fuertemente los falsos negativos y priorizar la detección de operaciones fraudulentas.

In [4]:
# Calcular escala de desbalanceo para XGBoost
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

print(f"Ratio de desbalanceo calculado (scale_pos_weight): {scale_weight:.2f}")

# Inicializar y entrenar el clasificador XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

print("\nEntrenando el modelo XGBoost... (esto tomará solo unos segundos)")
xgb_model.fit(X_train, y_train)
print("¡Entrenamiento finalizado con éxito!")

Ratio de desbalanceo calculado (scale_pos_weight): 336.34

Entrenando el modelo XGBoost... (esto tomará solo unos segundos)
¡Entrenamiento finalizado con éxito!


### 3. Evaluación de Rendimiento y Métricas de Negocio
Como lo establecimos en el EDA, evaluaremos el modelo utilizando la matriz de confusión, el reporte de clasificación y la curva **Precision-Recall (PR-AUC)**, métricas ideales para sistemas antifraude donde un falso negativo representa pérdidas financieras directas.

In [5]:
# Predicciones sobre el set de prueba
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

# Cálculo de PR-AUC
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
pr_auc_score = auc(recall, precision)

print(f"=== MÉTRICA CLAVE DE NEGOCIO ===")
print(f"PR-AUC Score: {pr_auc_score:.4f}\n")

print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, digits=4))

=== MÉTRICA CLAVE DE NEGOCIO ===
PR-AUC Score: 0.9973

=== REPORTE DE CLASIFICACIÓN ===
              precision    recall  f1-score   support

           0     1.0000    0.9997    0.9999    552439
           1     0.9115    0.9963    0.9520      1643

    accuracy                         0.9997    554082
   macro avg     0.9557    0.9980    0.9759    554082
weighted avg     0.9997    0.9997    0.9997    554082



> **🏆 Conclusión del Modelo de Machine Learning:**
> El clasificador XGBoost optimizado con pesos de clase y las variables de discrepancia financiera (*errorBalanceOrig* y *errorBalanceDest*) alcanzó un **Recall del 99.63%** y una **Precision del 91.15%** (F1-score de 0.9520). 
> * *Impacto Estratégico:* El modelo es capaz de blindar la plataforma de pagos detectando la casi totalidad de los fraudes reales, manteniendo al mismo tiempo una tasa de falsas alarmas extremadamente baja, cumpliendo con los estándares requeridos en la industria de tecnología financiera (*FinTech*).